# Download sample origins
Reads `passing_origins.parquet`, picks the first 10 rows with a known HEAD commit, and downloads their source trees using `swh-dataset-downloader`.

In [ ]:
import logging
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from downloader.models import DownloadSource, TaskStatus
from downloader.pipeline import run_pipeline

# ── Paths ─────────────────────────────────────────────────────────────────────
PARQUET      = Path("../output/passing_origins.parquet")  # adjust if needed
DOWNLOAD_DIR = Path("../output/sources")

# ── Tokens — loaded from python/.env (see .env.example) ──────────────────────
load_dotenv(Path(".") / ".env")
SWH_TOKEN    = os.environ.get("SWH_TOKEN", "")

assert SWH_TOKEN, "Create python/.env from .env.example and set SWH_TOKEN"

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("downloader").setLevel(logging.INFO)

In [2]:
df = pd.read_parquet(PARQUET)
print(f"Total passing origins: {len(df)}")
df[["url", "head_swhid", "top_language", "total_commits", "estimated_size_bytes"]].head()

Total passing origins: 132


,url,head_swhid,top_language,total_commits,estimated_size_bytes
0,https://github.com/freeCodeCamp/freeCodeCamp,swh:1:rev:222f0852dbf774058af95f172bf3f42c87e8...,TypeScript,38308,4422348
1,https://github.com/SerenityOS/serenity,swh:1:rev:80f586503daf0ecd0eac858e8b8e7e53b430...,C++,65189,53311868
2,https://github.com/nodejs/node,swh:1:rev:0c78a48d75008b87d874385a966a168a2b68...,JavaScript,94638,92996422
3,https://github.com/go-gitea/gitea,swh:1:rev:8cea1aeea5e72a7086c965c2197234f3928b...,Go,23214,26955310
4,https://github.com/kubernetes/kubernetes,swh:1:rev:347c54da6e90b238b9ba0c31d0f802d3109e...,Go,164515,60429384


In [3]:
# Keep only rows that have a resolved HEAD commit, then take the first 10
sample = (
    df[df["head_swhid"].str.len() > 0]
    .head(50)
    [["url", "head_swhid", "top_language", "total_commits"]]
    .reset_index(drop=True)
)
sample

,url,head_swhid,top_language,total_commits
0,https://github.com/freeCodeCamp/freeCodeCamp,swh:1:rev:222f0852dbf774058af95f172bf3f42c87e8...,TypeScript,38308
1,https://github.com/SerenityOS/serenity,swh:1:rev:80f586503daf0ecd0eac858e8b8e7e53b430...,C++,65189
2,https://github.com/nodejs/node,swh:1:rev:0c78a48d75008b87d874385a966a168a2b68...,JavaScript,94638
3,https://github.com/go-gitea/gitea,swh:1:rev:8cea1aeea5e72a7086c965c2197234f3928b...,Go,23214
4,https://github.com/kubernetes/kubernetes,swh:1:rev:347c54da6e90b238b9ba0c31d0f802d3109e...,Go,164515
5,https://github.com/k3s-io/k3s,swh:1:rev:441a42e8ce5a850e32546593ecad750a7ccc...,Go,88329
6,https://github.com/yt-dlp/yt-dlp,swh:1:rev:2685654a37141cca63eda3a92da0e2706e23...,Python,23188
7,https://github.com/etcd-io/etcd,swh:1:rev:21a976e9e254f3938f2ced3eb96310449ccf...,Go,26414
8,https://github.com/JuliaLang/julia,swh:1:rev:43ead4774248b1cf9cb84a349e9e90b3595c...,Julia,70017
9,https://github.com/keycloak/keycloak,swh:1:rev:603ecf20ebdc19067baa2200c6296912a56f...,Shell,29686


In [ ]:
pairs = [
    {"url": row.url, "swhid": row.head_swhid}
    for row in sample.itertuples()
]

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Jupyter supports top-level await — no asyncio.run() needed
results = await run_pipeline(
    pairs=pairs,
    output_base=DOWNLOAD_DIR,
    swh_token=SWH_TOKEN,
    debug=True,
)

18:37:34 [INFO] [0] skipped (already done)  https://github.com/freeCodeCamp/freeCodeCamp  222f0852dbf774058af95f172bf3f42c87e8ec8d
18:37:34 [INFO] [1] skipped (already done)  https://github.com/SerenityOS/serenity  80f586503daf0ecd0eac858e8b8e7e53b43065ae
18:37:34 [INFO] [2] skipped (already done)  https://github.com/nodejs/node  0c78a48d75008b87d874385a966a168a2b680b4d
18:37:34 [INFO] [3] skipped (already done)  https://github.com/go-gitea/gitea  8cea1aeea5e72a7086c965c2197234f3928b1779
18:37:34 [INFO] [4] skipped (already done)  https://github.com/kubernetes/kubernetes  347c54da6e90b238b9ba0c31d0f802d3109ed9e8
18:37:34 [INFO] [5] skipped (already done)  https://github.com/k3s-io/k3s  441a42e8ce5a850e32546593ecad750a7ccc3ab1
18:37:34 [INFO] [6] skipped (already done)  https://github.com/yt-dlp/yt-dlp  2685654a37141cca63eda3a92da0e2706e23ccfd
18:37:34 [INFO] [7] skipped (already done)  https://github.com/etcd-io/etcd  21a976e9e254f3938f2ced3eb96310449ccf8bef
18:37:34 [INFO] [8] skipped

CancelledError: 

In [ ]:
for task in results:
    source = task.source.value if task.source else "—"
    if task.status == TaskStatus.DONE:
        print(f"  OK   [{source:<9}] {task.url}")
        print(f"         → {task.output_dir}")
    else:
        print(f"  FAIL [{source:<9}] {task.url}")
        print(f"         ✗ {task.error}")

done   = sum(1 for t in results if t.status == TaskStatus.DONE)
failed = len(results) - done
print(f"\n{done}/{len(results)} succeeded  —  {failed} failed")

NameError: name 'results' is not defined